# Calling SolarFarmer **ModelChain** (2D) and **ModelChainAsync** (2D + 3D runs) endpoints

This notebook shows examples of how to run energy calculations in the cloud by calling the **ModelChain** and **ModelChainAsync** endpoints and how to easily access the calculation results all using the SolarFarmer Python SDK.

See details related to the **ModelChain** and **ModelChainAsync** endpoints and the outputs in SolarFarmer's public documentation:
- [Acquiring API key](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Introduction/ApiKey.html)
- [ModelChain endpoint](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Endpoints/ModelChainEndpoint.html)
- [ModelChainAsync endpoint](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Endpoints/ModelChainAsyncEndpoint.html)
- [API output results](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Endpoints/ModelChainEndpoint.html#outputs)
- [Cloud time-series results files](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/CalcRef/OutputFiles/CloudTestingFiles.html)
- [TerminateModelChainAsync](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Endpoints/TerminateModelChainAsyncEndpoint.html)



## 0. Prerequisites

**Notebook Information:**
- **Last Updated:** May 2026
- **Written for:** SolarFarmer SDK v0.2.0+
- [View latest version in repository](https://...)

### 0.1 Install the SolarFarmer Python SDK

This notebook requires the SolarFarmer Python SDK to be installed. Install it via pip:

```bash
pip install dnv-solarfarmer
```

In [1]:
import os
import solarfarmer as sf
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None
    print("NOTE: pandas is not installed. Section 3.3 (data accessor) examples will not work.")
    print("Install it with: pip install pandas\n")

# Check SDK version compatibility
NOTEBOOK_MIN_SDK_VERSION = "0.2.0"

print(f"SolarFarmer Python SDK v{sf.__version__}")

# Parse versions for comparison
def parse_version(v):
    """Simple version parser for x.y.z format"""
    return tuple(map(int, v.split('.')))

try:
    if parse_version(sf.__version__) < parse_version(NOTEBOOK_MIN_SDK_VERSION):
        print(f"\n    WARNING: This notebook requires SDK v{NOTEBOOK_MIN_SDK_VERSION} or later.")
        print(f"    Your version: {sf.__version__}")
        print(f"    Some examples may not work correctly.")
        print(f"    Upgrade with: pip install --upgrade solarfarmer\n")
except Exception:
    # If version parsing fails, just continue
    pass

sf.configure_logging();


SolarFarmer Python SDK v0.2.0


### 0.2 API Key Required

You need a SolarFarmer API key to run the calculations in this notebook. Instructions for acquiring one is [HERE](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Introduction/ApiKey.html)

**Important:** Avoid hardcoding your API key directly in notebook cells.

- **Use environment variables (Recommended):**

  The SDK automatically uses the `SF_API_KEY` environment variable. Set it in your terminal before starting Jupyter:

  **Linux/Mac:**
  ```bash
  export SF_API_KEY="your-key-here"
  ```

  **Windows:**
  ```bash
  set SF_API_KEY=your-key-here
  ```

- **Entering your API key (Alternative):**

  This notebook will prompt you to enter your API key and keep it hidden from view.

In [2]:
if os.getenv("SF_API_KEY") is None:
    print("WARNING: `SF_API_KEY` environment variable not set.")
    import getpass
    api_key = getpass.getpass("Enter your SolarFarmer API key: ")
    print("Using API key entered by user.\n")
else:
    api_key = os.getenv("SF_API_KEY")
    print("Using API key from environment variable `SF_API_KEY`\n")

Using API key from environment variable `SF_API_KEY`



### 0.3 Sample Data Required

This notebook uses sample input files (meteorological data, PAN and OND files, JSON configuration).

**Download the sample data:**

Clone the full repository:
```bash
git clone https://github.com/dnv-opensource/solarfarmer-python-sdk.git
```

Or download the `docs/notebooks/sample_data/` folder directly from the [repository](https://github.com/dnv-opensource/solarfarmer-python-sdk/):


The examples below assume the data is in a `sample_data/` folder relative to this notebook, but you can place it anywhere and update the path in the next cell.

In [3]:
# Path to the sample_data folder containing input files
# Modify this path if you have placed that data elsewhere
sample_data_dir = Path("sample_data")

# Verify the path exists
if not sample_data_dir.exists():
    print(f"WARNING: Sample data not found at: {sample_data_dir.absolute()}")
    print("Please download from `docs/notebooks/sample_data` in https://github.com/dnv-opensource/solarfarmer-python-sdk/")
    print("Or update the 'sample_data_dir' path above.")
else:
    print(f"Sample data found at: {sample_data_dir.absolute()}")

Sample data found at: c:\repos\solarfarmer-python-sdk-github\docs\notebooks\sample_data


## 1. ModelChain Endpoint

SolarFarmer API's sychronous modelchain endpoint can be invoked using the `run_energy_calculation()` function.

### 1.1 All inputs in a single folder

This example uses the files available in a single folder named "Inputs_Bern_2D_racks".

In [4]:
# A project ID parameter is optional and will be recorded in your SolarFarmer portal
# account for your own project management.
project_id = "Bern-CH SDK example"

# A folder with all required modelchain inputs (JSON, PAN, OND, meteorological data)
folder_with_inputs = sample_data_dir / "Inputs_Bern_2D_racks"

# Call the modelchain endpoint to run an energy calculation
results = sf.run_energy_calculation(
    api_key=api_key,  # This argument can be omitted if `SF_API_KEY` is set in your environment
    inputs_folder_path=folder_with_inputs,
    project_id=project_id,
    print_summary=True,
    save_outputs=False,
)

INFO: Making API call to ModelChain (2D) endpoint: https://solarfarmer.dnv.com/latest/api/ModelChain
Start time = 05-05-2026 12:13:41
INFO: SUCCESS: Calculation returned successfully (time taken: 6.4 seconds)


-------------------------------------------------------
		General site data:
-------------------------------------------------------
Project name = Bern-CH SDK example
Location (latitude, longitude, altitude) = 46.95507°, 7.37168°, 575.0 m
2D/3D = 2D calculation
Mounting type = Fixed-tilt racks
AC capacity of system = 0.375 MW
DC capacity of system = 0.4455 MW
Run in SolarFarmer API version 6.0.21
-------------------------------------------------------
    Annual Performance Summary (project year 1, 2017)
-------------------------------------  ---------------
Average annual temperature             10.61°C
Horizontal irradiation                 1192.73 kWh/m²
Irradiation on tilted plane            1373.40 kWh/m²
Effective irradiation on tilted plane  1293.84 kWh/m²
Energy yield                           1125.33 kWh/kWp
Net energy                             501.33 MWh/year
Performance Ratio                      0.8194
-------------------------------------  ---------------


### 1.2 Individual files and save the outputs

This example show how to specify all input files and the output directory.

**Saving Results**

The `save_outputs` parameter permits saving the output to a folder. 

If the parameter `outputs_folder_path` is not defined or is `None`, a new folder is created in the same directory as the JSON config file with the name `sf_outputs_<TIMESTAMP>`.


In [5]:
# Path to the JSON file
json_file = sample_data_dir / "EnergyCalcInputs2.json"

# Path to the meteorological file
met_file = sample_data_dir / "Bern-hour2.dat"

# Paths to the PAN files (it's a list)
pan_files = [
    folder_with_inputs / "CanadianSolar_CS6U-330M_APP.PAN"
]

# Paths to the OND files (it's a list)
ond_files = [
    folder_with_inputs / "Sungrow_SG125HV_APP.OND"
]

# Path to the horizon file (optional)
hor_file = None

# Folder to save the outputs (it will be created if it does not exist)
output_folder = sample_data_dir / "testing_output_123"

# Make the call to the API
results = sf.run_energy_calculation(
    project_id=project_id,
    energy_calculation_inputs_file_path=json_file,
    meteorological_data_file_path=met_file,
    pan_file_paths=pan_files,
    ond_file_paths=ond_files,
    horizon_file_path=hor_file,
    outputs_folder_path=output_folder,
    print_summary=True,
    save_outputs=True,
)

INFO: Making API call to ModelChain (2D) endpoint: https://solarfarmer.dnv.com/latest/api/ModelChain
Start time = 05-05-2026 12:13:50
INFO: SUCCESS: Calculation returned successfully (time taken: 5.1 seconds)


-------------------------------------------------------
		General site data:
-------------------------------------------------------
Project name = Bern-CH SDK example
Location (latitude, longitude, altitude) = 46.95507°, 7.37168°, 575.0 m
2D/3D = 2D calculation
Mounting type = Fixed-tilt racks
AC capacity of system = 0.375 MW
DC capacity of system = 0.4455 MW
Run in SolarFarmer API version 6.0.21
-------------------------------------------------------
    Annual Performance Summary (project year 1, 2017)
-------------------------------------  ---------------
Average annual temperature             10.61°C
Horizontal irradiation                 1192.73 kWh/m²
Irradiation on tilted plane            1373.40 kWh/m²
Effective irradiation on tilted plane  1293.84 kWh/m²
Energy yield                           1125.33 kWh/kWp
Net energy                             501.33 MWh/year
Performance Ratio                      0.8194
-------------------------------------  ---------------


## 2. ModelChainAsync Endpoint

SolarFarmer API's asynchronous modelchain endpoint can also be invoked using the `run_energy_calculation()` function.

`run_energy_calculation()` will determine whether the inputs provided require the asynchronous endpoint and call it.

In [6]:
# A project id (optional)
project_id = "Matera-IT SDK example"

# One folder with all the calculation inputs (JSON, PAN, OND, meteorological data)
folder_with_inputs = sample_data_dir / "Inputs_Matera_3D_trackers"

# Polling frequency to check the status of ModelchainAsync calculations (optional)
async_poll_time = 10

# Make the call to the API
results = sf.run_energy_calculation(
    inputs_folder_path=folder_with_inputs,
    project_id=project_id,
    async_poll_time=async_poll_time
)

INFO: Making API call to ModelChainAsync (3D) endpoint: https://solarfarmer.dnv.com/latest/api/ModelChainAsync
Start time = 05-05-2026 12:13:59
INFO: Instance ID returned = Matera-IT_SDK_example_2026-05-05T101359_u2wl4egqtl
INFO:  1: Time: 0:00:01 	Run status: Running 	Custom status: Calculation pending...  [Shading: 0.0%, ModelChain: 0.0%, Overall: 0.0%]
INFO: Plant stats: 3D Trackers, AC = 0.84 MW
INFO:  2: Time: 0:00:12 	Run status: Running 	Custom status: Calculation pending...  [Shading: 0.0%, ModelChain: 0.0%, Overall: 0.0%]
INFO:  3: Time: 0:00:22 	Run status: Running 	Custom status: Running model chain calculation (0/46 tasks complete)  [Shading: 100.0%, ModelChain: 0.0%, Overall: 35.3%]
INFO:  4: Time: 0:00:33 	Run status: Running 	Custom status: Running model chain calculation (0/46 tasks complete)  [Shading: 100.0%, ModelChain: 0.0%, Overall: 35.3%]
INFO:  5: Time: 0:00:44 	Run status: Running 	Custom status: Running post-processing (2/46 tasks complete)  [Shading: 100.0%, M

-------------------------------------------------------
		General site data:
-------------------------------------------------------
Project name = Matera-IT SDK example
Location (latitude, longitude, altitude) = 46.89330°, 7.36829°, 714.2 m
2D/3D = 3D calculation
Mounting type = Single-axis trackers
AC capacity of system = 0.84 MW
DC capacity of system = 1.197 MW
Run in SolarFarmer API version 6.0.21
-------------------------------------------------------
    Annual Performance Summary (project year 1, 2017)
-------------------------------------  ----------------
Average annual temperature             8.00°C
Horizontal irradiation                 1169.62 kWh/m²
Irradiation on tilted plane            1476.51 kWh/m²
Effective irradiation on tilted plane  1388.94 kWh/m²
Energy yield                           1194.57 kWh/kWp
Net energy                             1429.98 MWh/year
Performance Ratio                      0.8091
-------------------------------------  ----------------


## 3. Exploring the `CalculationResults` class

`run_energy_calculation` returns a `CalculationResults` object for both the `ModelChain` and `ModelChainAsync` endpoints.

This section covers four areas:

| # | Section | What it covers |
|---|---|---|
| 3.1 | **I/O** | Save results to disk; reload into a `CalculationResults` object |
| 3.2 | **Display methods** | Human-readable summaries printed to the console |
| 3.3 | **Data accessor methods** | `get_*` methods that return structured data for use with `pandas` |
| 3.4 | **Timeseries** | Hourly timeseries DataFrames and calculation metadata |

**Full method reference:**

| Method | Returns | Notes |
|---|---|---|
| `info()` / `get_info()` | — / `dict` | Availability of each data type in the object |
| `describe()` | — | Project specs + full annual performance summary |
| `performance()` / `get_performance()` | — / `dict` | Key energy metrics for a given project year |
| `print_annual_results()` / `get_annual_results_table()` | — / `dict` | Annual energy results and effects |
| `print_monthly_results()` / `get_monthly_results_table()` | — / `dict` | Monthly energy results and effects |
| `to_folder()` / `from_folder()` | — / `CalculationResults` | Persist and reload results |
| `loss_tree_timeseries()` | `pd.DataFrame` | Native SolarFarmer loss-tree timeseries |
| `pvsyst_timeseries()` | `pd.DataFrame` | PVsyst-format timeseries |
| `detailed_timeseries()` | `pd.DataFrame` | Extra intermediate steps (3D / async only) |
| `calculation_attributes()` | `dict` | Site, capacity, and API version metadata |


## 3.1 I/O — saving and loading results

Export results to disk with `.to_folder()` and reload them later with `.from_folder()`. This is useful for post-processing results separately from the API call, or for sharing results with others.

> **Note:** If the `CalculationResults` was loaded from a folder rather than directly from an API response, the exported time-series files will be missing some header and metadata lines. A message is shown when this applies.


In [7]:
output_folder = sample_data_dir / "testing_output_456"

results.to_folder(output_folder)

INFO: Results written out to sample_data\testing_output_456


In [8]:
results_data = sf.CalculationResults.from_folder(output_folder)

## 3.2 Display methods

The display methods print formatted output to the console. `describe()` is the most comprehensive — it combines project specifications and a full annual performance summary in one call, covering most of what `info()`, `performance()`, `print_annual_results()`, and `print_monthly_results()` show individually.


`describe()` prints project specifications and a full annual performance summary:


In [9]:
results_data.describe()

-------------------------------------------------------
		General site data:
-------------------------------------------------------
Project name = None
Location (latitude, longitude, altitude) = 46.89330°, 7.36829°, 714.2 m
2D/3D = 3D calculation
Mounting type = Single-axis trackers
AC capacity of system = 0.84 MW
DC capacity of system = 1.197 MW
Run in SolarFarmer API version 6.0.21
-------------------------------------------------------
    Annual Performance Summary (project year 1, 2017)
-------------------------------------  ----------------
Average annual temperature             8.00°C
Horizontal irradiation                 1169.62 kWh/m²
Irradiation on tilted plane            1476.51 kWh/m²
Effective irradiation on tilted plane  1388.94 kWh/m²
Energy yield                           1194.57 kWh/kWp
Net energy                             1429.98 MWh/year
Performance Ratio                      0.8091
-------------------------------------  ----------------


## 3.3 Programmatic data access

The `get_*` methods return structured data directly, making it straightforward to integrate results into custom analysis workflows with `pandas`, CSV/Excel exports, or other tools — without parsing printed tables.

> **Note:** `pandas` is an optional dependency and is not installed by default with `pip install solarfarmer`. The examples below require it:
> ```bash
> pip install pandas
> # or, to install all optional extras:
> pip install solarfarmer[all]
> ```


### 3.3.1 `get_info()` — availability metadata

Returns a `dict` describing which data types are present in the object. Useful for conditional checks in larger workflows.


In [10]:
info = results_data.get_info()
print(info)

# Use programmatically — e.g., guard before accessing timeseries
if info["has_loss_tree_timeseries"]:
    print("\nLoss tree timeseries is available.")


{'name': None, 'has_annual_data': True, 'has_monthly_data': True, 'has_calculation_attributes': True, 'has_loss_tree_timeseries': False, 'has_pvsyst_timeseries': False, 'has_detailed_timeseries': False}


### 3.3.2 `get_performance()` — key energy metrics

Returns key energy metrics for a specific project year as a `dict`. The example below also shows how to combine multiple years into a `pandas.DataFrame` for multi-year comparisons.


In [11]:
# Single year
perf_yr1 = results_data.get_performance(project_year=1)
print("Year 1 metrics:")
for key, val in perf_yr1.items():
    print(f"  {key}: {val}")

# Build a multi-year comparison DataFrame
num_years = len(results_data.AnnualData)
perf_df = pd.DataFrame([results_data.get_performance(y) for y in range(1, num_years + 1)])
print("\nMulti-year performance summary:")
perf_df


Year 1 metrics:
  project_year: 1
  calendar_year: 2017
  average_temperature: 8.0037424
  horizontal_irradiation: 1169.61585
  irradiation_on_tilted_plane: 1476.5072462
  effective_irradiation: 1388.9375277
  energy_yield: 1194.5706976
  net_energy: 1429.9760244
  performance_ratio: 0.8090517
  performance_ratio_bifacial: 0.8090517

Multi-year performance summary:


,project_year,calendar_year,average_temperature,horizontal_irradiation,irradiation_on_tilted_plane,effective_irradiation,energy_yield,net_energy,performance_ratio,performance_ratio_bifacial
0,1,2017,8.003742,1169.61585,1476.507246,1388.937528,1194.570698,1429.976024,0.809052,0.809052


### 3.3.3 `get_annual_results_table()` — annual energy results and effects

Returns a `dict` with `"energy_results"` and `"effects"` keys, each containing a list of dicts (one per year) that are directly `pd.DataFrame`-constructible and can be exported to CSV or Excel.


In [12]:
annual_table = results_data.get_annual_results_table()

# Energy results — one row per year
energy_df = pd.DataFrame(annual_table["energy_results"])
print("Annual energy results:")
display(energy_df[["project_year", "calendar_year", "ghi", "gi", "net_energy", "energy_yield", "performance_ratio"]])

# Effects — one row per year (fractional losses, e.g. -0.02 = -2 %)
effects_df = pd.DataFrame(annual_table["effects"])
print("\nAnnual effects (fraction):")
display(effects_df[["project_year", "calendar_year", "soiling", "angular", "temperature", "ohmic_dc", "ohmic_ac"]])


Annual energy results:


,project_year,calendar_year,ghi,gi,net_energy,energy_yield,performance_ratio
0,1,2017,1169.61585,1476.507246,1429.976024,1194.570698,0.809052



Annual effects (fraction):


,project_year,calendar_year,soiling,angular,temperature,ohmic_dc,ohmic_ac
0,1,2017,-0.01,-0.021228,-0.026427,0,0


### 3.3.4 `get_monthly_results_table()` — monthly energy results and effects

Same structure as `get_annual_results_table()` but with one row per month per year. The `project_years` argument accepts either 1-based project year indices (e.g. `[1, 2]`) or actual calendar years (e.g. `[2020, 2021]`).


In [13]:
monthly_table = results_data.get_monthly_results_table(project_years=[1])
monthly_df = pd.DataFrame(monthly_table["energy_results"])

# Key monthly metrics
print("Monthly energy results (project year 1):")
display(monthly_df[["month_name", "ghi", "gi", "net_energy", "energy_yield", "performance_ratio"]])


Monthly energy results (project year 1):


,month_name,ghi,gi,net_energy,energy_yield,performance_ratio
0,Jan,35.57600,43.498055,43.960213,36.723955,0.844267
1,Feb,62.34360,80.221165,82.605353,69.011741,0.860268
2,Mar,90.31100,108.734642,106.783218,89.208901,0.820428
3,Apr,100.81500,123.231076,117.482462,98.147723,0.796453
4,May,152.71100,188.280098,177.962677,148.674325,0.789644
5,Jun,146.08960,182.257392,171.768287,143.498735,0.787341
6,Jul,168.60545,215.948287,204.351455,170.719171,0.790556
7,Aug,168.52665,225.541330,216.086098,180.526844,0.800416
8,Sep,107.58165,140.071021,138.607432,115.794276,0.826683
9,Oct,73.16145,94.822994,95.806319,80.038990,0.844088


## 3.4 Timeseries data

Three `pd.DataFrame` timeseries and a calculation attributes `dict` are accessible via dedicated accessor methods. Each will be `None` if the API did not return that data for the given calculation type (e.g., the detailed timeseries is only available for 3D / async runs).

| Method | Contents |
|---|---|
| `loss_tree_timeseries()` | Native SolarFarmer loss-tree timeseries |
| `pvsyst_timeseries()` | Timeseries formatted in PVsyst style |
| `detailed_timeseries()` | Extra intermediate modelling steps; useful for diagnostics |
| `calculation_attributes()` | Metadata dict: site location, capacities, API version, etc. |


In [14]:
loss_tree_df = results_data.loss_tree_timeseries()
pvsyst_df = results_data.pvsyst_timeseries()
detailed_df = results_data.detailed_timeseries()
calc_attrs = results_data.calculation_attributes()

# Report what is available and show a preview
for name, obj in [
    ("Loss tree timeseries", loss_tree_df),
    ("PVsyst timeseries", pvsyst_df),
    ("Detailed timeseries", detailed_df),
]:
    if obj is not None:
        print(f"{name}: {len(obj)} rows × {len(obj.columns)} columns")
        display(obj.head(3))
    else:
        print(f"{name}: not available")

# Calculation attributes — site and system metadata
if calc_attrs is not None:
    sys_attrs = calc_attrs["systemAttributes"]
    print(
        f"\nCalculation attributes: {sys_attrs['dcCapacityInMegawatts']} MWp DC, "
        f"{sys_attrs['acCapacityInMegawatts']} MWac, "
        f"SF API v{sys_attrs['solarFarmerApiVersion']}"
    )


Loss tree timeseries: not available
PVsyst timeseries: not available
Detailed timeseries: not available

Calculation attributes: 1.197 MWp DC, 0.84 MWac, SF API v6.0.21
